###  Architecture 
Here instead of EfficientNet CNN based model, Swin transformer is used to analyze the inclusion of metadata and their interaction to the images. All the hyper parameters kept same except the batch_size to 12 instead of 32 due to heavy computation in swin model.



In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath("../.."))

In [2]:
# Cell 1: imports & setup
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

gc.enable()

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

from src.config import EXP_CONFIGS
from src.utils import set_seed, save_config
from src.data import build_transforms, ImageTabDataset
from src.models import EarlyFusionNet



main_folder = "../.."
data_csv = os.path.join(main_folder, "data", "train.csv")
img_folder = os.path.join(main_folder, "data", "train")
out_dir = os.path.join(main_folder, "outputs", "extra","exp5_SwinT_StrongAug_LinearHead")

df = pd.read_csv(data_csv)
cfg = EXP_CONFIGS["exp5"]   # define in config
cfg["head_type"] = "linear"
cfg["name"] = "exp5_SwinT_StrongAug_LinearHead"
TARGET = "Pawpularity"
tab_cols = [c for c in df.columns if c not in ["Id", TARGET]]

os.makedirs(out_dir, exist_ok=True)
save_config(cfg, out_dir)
set_seed(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


/home/ghias/miniconda3/envs/rapids-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from src.train import run_single_fold

kf = KFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["seed"])
oof_pred = np.zeros(len(df))
oof_true = df[TARGET].values
fold_index = np.full(len(df), -1, dtype=int)
fold_rmse = []

start_all = time.time()
for fold, (tr_idx, val_idx) in enumerate(kf.split(df), start=1):
    print(f"\n=== {cfg['name']}: Fold {fold} ===")
    train_df = df.iloc[tr_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    best_rmse, val_preds, val_targets, val_ids = run_single_fold(
    fold=fold,
    train_df=train_df,
    val_df=val_df,
    img_folder=img_folder,
    cfg=cfg,
    out_dir=out_dir,
    device=device,
    mode="fusion",      #  fusion
    tab_cols=tab_cols,
    workers=16,
    pin_memory=True,
    persistent_workers=True,
    )

    oof_pred[val_idx] = val_preds
    fold_index[val_idx] = fold
    fold_rmse.append(best_rmse)
    print(f"Fold {fold} best RMSE: {best_rmse:.4f}")

all_sec = time.time() - start_all
print(f"\nTotal training time: {int(all_sec//60)}m {int(all_sec%60)}s")
# final metrics + OOF with fold column
oof_rmse = root_mean_squared_error(oof_true, oof_pred)
fold_rmse = np.array(fold_rmse)
print(f"\nOOF RMSE: {oof_rmse:.4f}")
print(f"Fold RMSEs: {fold_rmse.tolist()}  Mean={fold_rmse.mean():.4f}  Std={fold_rmse.std():.4f}")

oof_df = pd.DataFrame({
    "Id": df["Id"],
    "fold": fold_index,      #  fold for each sample
    "ytrue": oof_true,
    "oof_pred": oof_pred,
})
oof_df["abs_err"] = (oof_df["ytrue"] - oof_df["oof_pred"]).abs()
oof_df.to_csv(os.path.join(out_dir, "oof_detail.csv"), index=False)
oof_df.sort_values("abs_err", ascending=False).head(50).to_csv(
    os.path.join(out_dir, "top50_errors.csv"), index=False
)

np.save(os.path.join(out_dir, "oof_pred.npy"), oof_pred)
np.save(os.path.join(out_dir, "fold_rmse.npy"), fold_rmse)
with open(os.path.join(out_dir, "metrics.txt"), "w") as f:
    f.write(f"OOF_RMSE: {oof_rmse:.4f}\n")
    f.write(f"Fold_RMSE: {fold_rmse.tolist()}\nMean: {fold_rmse.mean():.4f}\nStd: {fold_rmse.std():.4f}\n")



=== exp5_SwinT_StrongAug_LinearHead: Fold 1 ===
Epoch 1/10 | Fold 1 | Train[BCE]: Loss=0.6495 | ValRMSE: 18.0207
Epoch 2/10 | Fold 1 | Train[BCE]: Loss=0.6392 | ValRMSE: 17.6690
Epoch 3/10 | Fold 1 | Train[BCE]: Loss=0.6327 | ValRMSE: 17.9845
Epoch 4/10 | Fold 1 | Train[BCE]: Loss=0.6236 | ValRMSE: 17.9874
Epoch 5/10 | Fold 1 | Train[BCE]: Loss=0.6133 | ValRMSE: 18.1768
Epoch 6/10 | Fold 1 | Train[BCE]: Loss=0.6045 | ValRMSE: 18.5429
Epoch 7/10 | Fold 1 | Train[BCE]: Loss=0.5973 | ValRMSE: 18.4535
Early stopping at epoch 7
Fold 1 best RMSE: 17.6690

=== exp5_SwinT_StrongAug_LinearHead: Fold 2 ===
Epoch 1/10 | Fold 2 | Train[BCE]: Loss=0.6486 | ValRMSE: 18.1002
Epoch 2/10 | Fold 2 | Train[BCE]: Loss=0.6396 | ValRMSE: 18.8309
Epoch 3/10 | Fold 2 | Train[BCE]: Loss=0.6322 | ValRMSE: 18.3640
Epoch 4/10 | Fold 2 | Train[BCE]: Loss=0.6235 | ValRMSE: 18.5701
Epoch 5/10 | Fold 2 | Train[BCE]: Loss=0.6135 | ValRMSE: 19.0643
Epoch 6/10 | Fold 2 | Train[BCE]: Loss=0.6042 | ValRMSE: 18.8245
Early

In [4]:
import pandas
df = pd.read_csv(out_dir+"/oof_detail.csv")

for fold in range(1, 6):
    sub = df[df.fold == fold]
    rmse = ((sub.ytrue - sub.oof_pred)**2).mean() ** 0.5
    print(f"Fold {fold} OOF RMSE: {rmse:.4f} (n={len(sub)})")


Fold 1 OOF RMSE: 17.5889 (n=1983)
Fold 2 OOF RMSE: 17.7573 (n=1983)
Fold 3 OOF RMSE: 17.2976 (n=1982)
Fold 4 OOF RMSE: 17.6068 (n=1982)
Fold 5 OOF RMSE: 17.7455 (n=1982)


In [ ]:
from src.plot import plot_all_folds_history
plot_all_folds_history(out_dir, folds=[1,2,3,4,5], title_prefix="Exp5")

In [ ]:
#  Inspect worst error rows

errors_path = os.path.join(out_dir, "top50_errors.csv")
err_df = pd.read_csv(errors_path)
err_df.head()

In [ ]:
import os
import pandas as pd
from src.plot import (
   plot_oof_true_pred_lines,
    show_images_grid,
)

# Load OOF detail and top-errors
oof_df = pd.read_csv(os.path.join(out_dir, "oof_detail.csv"))
err_df = pd.read_csv(os.path.join(out_dir, "top50_errors.csv"))

# Plots
plot_oof_true_pred_lines(oof_df, title_prefix="Exp5")





In [ ]:
from src.plot import compare_oof_by_bins
bins = [0, 20, 40, 60, 80, 100]

df_bins, rmse3_all, rmse5_all, gap = compare_oof_by_bins(
    oof_path_a="../outputs/exp3/oof_detail.csv",
    oof_path_b="../outputs/exp5/oof_detail.csv",
    bins=bins,
    label_a="Exp3_Swin_NoFusion",
    label_b="Exp5_Swin_EarlyFusion",
    title_prefix="Swin",
)

In [ ]:
df_bins.head()

In [ ]:
# Images of worst errors
img_folder = os.path.join(main_folder, "data", "train")
show_images_grid(err_df, img_folder, n=12, title_prefix="exp5_swin_earlyFusion_top_errors")

In [ ]:

show_images_grid(oof_df, img_folder, n=12, title_prefix="exp5_swin_earlyFusion_first 12 images")